# Robustez Menendez

Notebook principal para esta fase del TFG.

Sirve para:

- ejecutar la suite real de variantes Menendez
- ver donde muere el embudo de senales
- inspeccionar bloqueos y estados de setup
- revisar snapshots de indicadores
- comparar de forma rapida contra benchmarks simples


In [ ]:
from pathlib import Path
import sys
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 200)

cwd = Path.cwd().resolve()
root = next((path for path in [cwd, *cwd.parents] if (path / 'backtests' / 'menendez' / 'menendez_pipeline.py').exists()), None)
if root is None:
    raise RuntimeError('No se encontro la raiz del proyecto con menendez_pipeline.py')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from backtests.menendez.menendez_pipeline import (
    ejecutar_suite_experimental_menendez,
    extraer_indicator_snapshot,
    colorear_comparativa,
)
from backtests.benchmarks.simple_benchmarks import ejecutar_benchmarks
from backtests.benchmarks.statistics import bootstrap_por_estrategia


## Configuracion

Puedes empezar con pocos simbolos para depurar mejor y despues ampliar a todo el grupo.

In [ ]:
symbols = None
# Ejemplos:
symbols = ['EURUSD.r']
# symbols = ['EURUSD.r', 'GBPUSD.r', 'USDJPY.r']

variant_names = [
    'faithful_operable_trigger_or',
    'faithful_operable_sma200_primary',
    'experimental_composite_x',
]

use_cache = False
force_rebuild = True
use_disk_cache = False
parallel = True


In [ ]:
suite = ejecutar_suite_experimental_menendez(
    symbols=symbols,
    variant_names=variant_names,
    verbose=True,
    use_cache=use_cache,
    force_rebuild=force_rebuild,
    use_disk_cache=use_disk_cache,
    parallel=parallel,
)

suite['summary_table']


In [ ]:
if not suite['summary_table'].empty:
    display(colorear_comparativa(suite['summary_table']))
else:
    print('No hay resumen disponible.')


## Elegir variante a inspeccionar

Cambia `variant_name` para revisar con detalle una variante concreta.

In [ ]:
variant_name = 'faithful_operable_sma200_primary'
bundle = suite['variants'][variant_name]

bundle['run_meta']


In [ ]:
bundle['summary_metrics']


In [ ]:
bundle['signal_funnel']['stage_counts']


In [ ]:
bundle['signal_funnel']['block_reasons']


In [ ]:
bundle['signal_funnel']['status_distribution']


In [ ]:
bundle['screener_rows'].head(30)


In [ ]:
bundle['trade_log'].head(30)


## Snapshot de indicadores

Si hay setups o filas activas, puedes inspeccionar un simbolo concreto alrededor de una barra.

In [ ]:
portfolio = bundle['context_df']
symbols_available = sorted(portfolio.keys())
symbols_available[:10]


In [ ]:
symbol_to_inspect = symbols_available[0] if symbols_available else None

if symbol_to_inspect is None:
    print('No hay simbolos cargados en el bundle.')
else:
    snapshot = extraer_indicator_snapshot(
        portfolio,
        symbol=symbol_to_inspect,
        bars_before=8,
        bars_after=8,
    )
    display(snapshot)


## Benchmarks simples

Usan el mismo stack monetario para que la comparacion con Menendez sea defendible.

In [ ]:
benchmark_result = ejecutar_benchmarks(
    bundle['context_df'],
    return_details=True,
    parallel=parallel,
)

benchmark_result['summary_metrics']


In [ ]:
bootstrap_por_estrategia(benchmark_result, n_boot=200).head(50)


## Lectura recomendada

Orden sugerido para sacar conclusiones:

1. `suite['summary_table']`
2. `block_reasons` de `faithful_strict`
3. `status_distribution` de `faithful_strict`
4. comparar con la mejor `faithful_operable_*`
5. mirar `screener_rows` y `indicator_snapshot`
6. despues comparar con benchmarks


In [ ]:
bundle = suite["variants"][variant_name]
df = bundle["context_df"]["EURUSD.r"]

cols = [
    "H4_SOURCE_TIME",
    "H4_ATTRACTOR_DIR",
    "H4_ATTRACTOR_STAGE",
    "H4_ATTRACTOR_BLOCK_REASON",
    "H4_PSAR_POLARITY",
    "H4_BULLISH_FAN",
    "H4_BEARISH_FAN",
    "H4_SMA_21_SLOPE",
    "H4_MACD_HIST",
    "H4_MACD_NEUTRAL",
    "H4_STANDBY",
]

df[cols].drop_duplicates().head(50)


In [ ]:
df["H4_ATTRACTOR_BLOCK_REASON"].value_counts(dropna=False)
